# 조합 5 GPT 경로 레포 코드 검증

`feat/combo5-gpt` 의 `pipeline._run_prompt_mode` 를 직접 불러 노트북 v9 결과와 같은지 본다.
GPT 경로는 InsightFace(CPU)·MediaPipe(CPU)만 쓰므로 CPU 런타임으로 돌린다. 조합 3·SDXL 은 올리지 않는다.
볼 곳: 2048 결과 크기, 턱선 윤곽, 앞머리·눈썹, 헤어 픽셀이 원본과 같은지, 장당 시간.

In [ ]:
%cd /content
import importlib
import os
import shutil
import sys
import time
from getpass import getpass
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
from google.colab import drive
from PIL import Image

drive.mount("/content/drive")

REPO = "/content/SalonCutAI"
shutil.rmtree(REPO, ignore_errors=True)
!git clone -q -b feat/combo5-gpt https://github.com/qja0707/SalonCutAI.git {REPO}
!pip install -q diffusers==0.39.0 transformers==5.14.1 peft==0.19.1 accelerate==1.14.0 \
    insightface onnxruntime mediapipe==1.0.0 opencv-contrib-python-headless \
    facexlib torchvision openai==2.53.0

os.environ["IMAGE_GEN_ENABLED"] = "1"
os.environ["SALON_STORAGE_DIR"] = "/content/storage"
os.environ["OPENAI_KEY"] = getpass("OpenAI API key: ")
sys.path.insert(0, f"{REPO}/backend")
importlib.invalidate_caches()

import mediapipe as mp
from src.ai_engine.image_gen import downloads, loader, pipeline, settings, storage
from src.schemas.face_swap import FacePromptOptions

downloads.ensure_models()
print("missing:", downloads.missing_files())
print("engine:", settings.PROMPT_MODE_ENGINE, settings.GPT_IMAGE_MODEL)

loader.get_face_app()
loader.get_landmarker()
loader.get_segmenter()
print("준비 완료")

In [ ]:
SALON = Path("/content/drive/MyDrive/saloncut_data/test_images/salon")
NORMAL = Path("/content/drive/MyDrive/saloncut_data/test_images/normal")
OUT = Path("/content/drive/MyDrive/saloncut_data/outputs/combo5_gpt_repo_0826")
OUT.mkdir(parents=True, exist_ok=True)

CASES = [
    ("salon_01_long_wave_brown", SALON, "여성", "20대"),
    ("normal_09_dark_skin", NORMAL, "남성", "20대"),
]
STYLES = {"puppy": "강아지상", "cat": "고양이상", "fox": "여우상"}


def opts(gender, age, style):
    prompt = FacePromptOptions(
        ethnicity="한국인", gender=gender, age=age, face_style=style, expression="무표정"
    )
    return SimpleNamespace(face=SimpleNamespace(mode="prompt", prompt=prompt))


def hair_class(img):
    res = loader.get_segmenter().segment(
        mp.Image(image_format=mp.ImageFormat.SRGB, data=np.array(img))
    )
    return np.squeeze(res.category_mask.numpy_view()) == 1


for name, folder, gender, age in CASES:
    src = storage.to_stored_size(Image.open(folder / f"{name}.jpg").convert("RGB"))
    hair = hair_class(src)
    fig, axes = plt.subplots(1, len(STYLES) + 1, figsize=(6 * (len(STYLES) + 1), 7))
    axes[0].imshow(src); axes[0].set_title(f"{name[:9]} src {src.size}", fontsize=11)
    for ax, (key, style) in zip(axes[1:], STYLES.items()):
        t = time.time()
        fin = pipeline._run_prompt_mode(src, opts(gender, age, style), seed=0)
        sec = time.time() - t
        fin.save(OUT / f"{name}_{key}.png")
        diff = np.abs(np.array(src).astype(np.int16) - np.array(fin).astype(np.int16)).max(axis=2) > 8
        ax.imshow(fin); ax.set_title(f"{key} {fin.size} {sec:.0f}s", fontsize=11)
        print(f"{name}  {key}  {sec:.0f}s  size {fin.size}  헤어 안 바뀐 픽셀 {(diff & hair).sum()}")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(OUT / f"grid_{name}_repo.png", dpi=100, bbox_inches="tight")
    plt.show()

In [ ]:
from src.ai_engine.image_gen import combo5_gpt, masks

CHECK = [("salon_01_long_wave_brown", SALON, "여성", "20대", "여우상", "jaw"),
         ("normal_09_dark_skin", NORMAL, "남성", "20대", "강아지상", "hair")]
ERODES = [0.02, 0.03, 0.04]

fig, axes = plt.subplots(len(CHECK), len(ERODES) + 1, figsize=(6 * (len(ERODES) + 1), 7 * len(CHECK)))
for i, (name, folder, gender, age, style, region) in enumerate(CHECK):
    src = storage.to_stored_size(Image.open(folder / f"{name}.jpg").convert("RGB"))
    full, face_d, _ = combo5_gpt.generate(src, opts(gender, age, style).face.prompt)
    _, fw = combo5_gpt._face_box(src)
    x1, y1, x2, y2 = combo5_gpt._face_box(src)[0]
    crop = ((x1 - fw // 4, y1 + fw // 3, x2 + fw // 4, y2 + fw // 3) if region == "jaw"
            else (x1 - fw // 4, max(0, y1 - fw // 2), x2 + fw // 4, y1 + fw // 2))
    axes[i, 0].imshow(src.crop(crop)); axes[i, 0].set_title(f"{name[:9]} src", fontsize=12)
    for ax, e in zip(axes[i, 1:], ERODES):
        settings.HAIR_ERODE_RATIO = e
        hair = masks.build_hair_mask_gpt(src, fw)
        fin = pipeline._postprocess_gpt(src, full, face_d, hair)
        fin.save(OUT / f"{name}_{style}_erode{e}.png")
        ax.imshow(fin.crop(crop)); ax.set_title(f"erode {e}", fontsize=12)
settings.HAIR_ERODE_RATIO = 0.02
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUT / "grid_erode_recheck.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
from src.ai_engine.image_gen import prompt_map

FOX = {
    "A 현재": prompt_map.STYLE_SENT["여우상"],
    "B 턱 삭제": (
        "a fox-like face: long narrow eyes with slightly lowered lids and outer "
        "corners stretched upward, and a thin straight nose"
    ),
    "C 턱 폭 유지": (
        "a fox-like face: long narrow eyes with slightly lowered lids and outer "
        "corners stretched upward, a thin straight nose, and a softly tapered chin "
        "that keeps the original jaw width"
    ),
}
FOX_CASES = [("salon_01_long_wave_brown", SALON), ("salon_03_long_wave_ring", SALON)]

fig, axes = plt.subplots(len(FOX_CASES), len(FOX) + 1, figsize=(6 * (len(FOX) + 1), 7 * len(FOX_CASES)))
for i, (name, folder) in enumerate(FOX_CASES):
    src = storage.to_stored_size(Image.open(folder / f"{name}.jpg").convert("RGB"))
    axes[i, 0].imshow(src); axes[i, 0].set_title(f"{name[:9]} src", fontsize=12)
    for ax, (label, sent) in zip(axes[i, 1:], FOX.items()):
        prompt_map.STYLE_SENT["여우상"] = sent
        fin = pipeline._run_prompt_mode(src, opts("여성", "20대", "여우상"), seed=0)
        fin.save(OUT / f"{name}_fox_{label[0]}.png")
        ax.imshow(fin); ax.set_title(label, fontsize=12)
prompt_map.STYLE_SENT["여우상"] = FOX["A 현재"]
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUT / "grid_fox_sentence.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
src = storage.to_stored_size(Image.open(SALON / "salon_01_long_wave_brown.jpg").convert("RGB"))
(x1, y1, x2, y2), _ = combo5_gpt._face_box(src)
fw = x2 - x1
jaw = (x1 - fw // 4, y1 + fw // 3, x2 + fw // 4, y2 + fw // 3)

DIL = [0.10, 0.14, 0.18]
fig, axes = plt.subplots(1, len(DIL) + 1, figsize=(6 * (len(DIL) + 1), 7))
axes[0].imshow(src.crop(jaw)); axes[0].set_title("src", fontsize=12)
for ax, d in zip(axes[1:], DIL):
    settings.GPT_EDIT_DILATE_RATIO = d
    settings.GPT_FACE_DILATE_RATIO = d
    fin = pipeline._run_prompt_mode(src, opts("여성", "20대", "여우상"), seed=0)
    fin.save(OUT / f"salon_01_fox_editdilate{d}.png")
    ax.imshow(fin.crop(jaw)); ax.set_title(f"edit/face dilate {d}", fontsize=12)
settings.GPT_EDIT_DILATE_RATIO = 0.10
settings.GPT_FACE_DILATE_RATIO = 0.10
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUT / "grid_fox_editdilate.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
import cv2
from PIL import ImageFilter

src = storage.to_stored_size(Image.open(SALON / "salon_01_long_wave_brown.jpg").convert("RGB"))
o = opts("여성", "20대", "여우상").face.prompt
face_mask = masks.build_face_mask(src)
box, fw_face = combo5_gpt._face_box(src)
x1, y1, x2, y2 = box
size = settings.GPT_CROP_SIZE
edit_face = masks.dilate_mask(face_mask, fw_face, settings.GPT_EDIT_DILATE_RATIO)
edit_mask = masks.build_gen_mask(edit_face, masks.build_hair_mask(src))
crop = src.crop(box).resize((size, size), Image.LANCZOS)
m = edit_mask.crop(box).resize((size, size), Image.NEAREST).convert("L")
m_np = np.array(m) > 127
prompt = prompt_map.build_face_sentence(o)

def fill_blur(c):
    return Image.composite(c.filter(ImageFilter.GaussianBlur(40)), c, m)

def fill_mean(c):
    arr = np.array(c).copy()
    arr[m_np] = arr[m_np].mean(axis=0).astype(np.uint8)
    return Image.fromarray(arr)

face_d = masks.dilate_mask(face_mask, fw_face, settings.GPT_FACE_DILATE_RATIO)
hair = masks.build_hair_mask_gpt(src, fw_face)
fw = x2 - x1
jaw = (x1 - fw // 4, y1 + fw // 3, x2 + fw // 4, y2 + fw // 3)

VARS = [("원본 그대로", crop), ("블러 채움", fill_blur(crop)), ("평균색 채움", fill_mean(crop))]
fig, axes = plt.subplots(2, len(VARS) + 1, figsize=(6 * (len(VARS) + 1), 13))
axes[0, 0].imshow(src.crop(jaw)); axes[0, 0].set_title("src", fontsize=12)
axes[1, 0].axis("off")
for j, (label, sent_crop) in enumerate(VARS, start=1):
    rgba = sent_crop.copy().convert("RGBA")
    rgba.putalpha(Image.fromarray(255 - np.array(m)))
    raw = combo5_gpt._edit(sent_crop, rgba, prompt)
    full = src.copy()
    full.paste(raw.resize((x2 - x1, y2 - y1), Image.LANCZOS), (x1, y1))
    fin = pipeline._postprocess_gpt(src, full, face_d, hair)
    fin.save(OUT / f"salon_01_fox_fill{j}.png")
    axes[0, j].imshow(sent_crop); axes[0, j].set_title(f"보낸 크롭 — {label}", fontsize=12)
    axes[1, j].imshow(fin.crop(jaw)); axes[1, j].set_title(f"결과 — {label}", fontsize=12)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUT / "grid_fox_fill.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
diff = np.abs(np.array(src).astype(np.int16) - np.array(fin).astype(np.int16)).max(axis=2) > 8
ov = np.array(src).copy()
ov[diff] = (ov[diff] * 0.4 + np.array([255, 0, 0]) * 0.6).astype(np.uint8)
for m_, color in [(hair, (0, 255, 0)), (face_d, (0, 128, 255))]:
    cnts, _ = cv2.findContours(np.array(m_), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(ov, cnts, -1, color, 3)

fig, axes = plt.subplots(1, 4, figsize=(24, 7))
for ax, (lab, im) in zip(axes, [("src", src), ("raw pasted", full), ("final", fin), ("changed=red hair=green face+10%=blue", Image.fromarray(ov))]):
    ax.imshow(im.crop(jaw)); ax.set_title(lab, fontsize=11); ax.axis("off")
plt.tight_layout()
plt.savefig(OUT / "grid_fox_fill_diag.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
def hair_minus_oval(hair_mask, oval_mask):
    h = np.array(hair_mask).copy()
    h[np.array(oval_mask) > 127] = 0
    return Image.fromarray(h)

oval_e = masks._morph(face_mask, cv2.MORPH_ERODE, int(fw_face * 0.02))
VARS = [("현재", hair), ("헤어 − oval", hair_minus_oval(hair, face_mask)),
        ("헤어 − oval 침식 2%", hair_minus_oval(hair, oval_e))]
fig, axes = plt.subplots(1, len(VARS) + 1, figsize=(6 * (len(VARS) + 1), 7))
axes[0].imshow(src.crop(jaw)); axes[0].set_title("src", fontsize=12)
for ax, (lab, hm_) in zip(axes[1:], VARS):
    f = pipeline._postprocess_gpt(src, full, face_d, hm_)
    f.save(OUT / f"salon_01_fox_hairoval_{lab}.png")
    ax.imshow(f.crop(jaw)); ax.set_title(lab, fontsize=12)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUT / "grid_fox_hair_oval.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
settings.HAIR_CLOSE_RATIO, settings.HAIR_ERODE_RATIO, settings.BROW_EXCLUDE_RATIO = 0.123, 0.032, 0.048
hair16 = masks.build_hair_mask_gpt(src, fw_face)
face16 = masks.dilate_mask(face_mask, fw_face, 0.16)
fin16 = pipeline._postprocess_gpt(src, full, face16, hair16)
settings.HAIR_CLOSE_RATIO, settings.HAIR_ERODE_RATIO, settings.BROW_EXCLUDE_RATIO = 0.077, 0.02, 0.03

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
for ax, (lab, im) in zip(axes, [("src", src), ("현재 비율", fin), ("환산 비율(×1.6)", fin16)]):
    ax.imshow(im.crop(jaw)); ax.set_title(lab, fontsize=12); ax.axis("off")
plt.tight_layout()
plt.savefig(OUT / "grid_fox_ratio16.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
CONV = dict(GPT_EDIT_DILATE_RATIO=0.16, GPT_FACE_DILATE_RATIO=0.16,
            HAIR_CLOSE_RATIO=0.123, HAIR_ERODE_RATIO=0.032, BROW_EXCLUDE_RATIO=0.048)
ORIG = {k: getattr(settings, k) for k in CONV}
for k, v in CONV.items():
    setattr(settings, k, v)
fin_conv = pipeline._run_prompt_mode(src, opts("여성", "20대", "여우상"), seed=0)
fin_conv.save(OUT / "salon_01_fox_conv16.png")
for k, v in ORIG.items():
    setattr(settings, k, v)

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
for ax, (lab, im) in zip(axes, [("src", src), ("현재", fin), ("전부 ×1.6 환산", fin_conv)]):
    ax.imshow(im.crop(jaw)); ax.set_title(lab, fontsize=12); ax.axis("off")
plt.tight_layout()
plt.savefig(OUT / "grid_fox_conv16.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
from PIL import ImageFilter

def hair_only(src, full, hair_mask, feather=0):
    W, H = src.size
    hb = hair_mask.filter(ImageFilter.GaussianBlur(settings.HAIR_BLUR * max(W, H) / 1024))
    if feather <= 0:
        return Image.composite(src, full, hb)
    # 크롭 상자 안쪽만 GPT, 테두리는 페더링
    box_m = Image.new("L", (W, H), 0)
    inset = int((x2 - x1) * feather)
    box_m.paste(255, (x1 + inset, y1 + inset, x2 - inset, y2 - inset))
    box_m = box_m.filter(ImageFilter.GaussianBlur(inset))
    comp = Image.composite(full, src, box_m)
    return Image.composite(src, comp, hb)

VARS = [("현재(얼굴 마스크 재합성)", fin), ("헤어만 되돌림", hair_only(src, full, hair)),
        ("헤어만 + 크롭 페더 5%", hair_only(src, full, hair, feather=0.05))]
fig, axes = plt.subplots(2, len(VARS) + 1, figsize=(6 * (len(VARS) + 1), 14))
axes[0, 0].imshow(src.crop(jaw)); axes[0, 0].set_title("src 턱", fontsize=12)
axes[1, 0].imshow(src); axes[1, 0].set_title("src 전체", fontsize=12)
for j, (lab, im) in enumerate(VARS, start=1):
    im.save(OUT / f"salon_01_fox_haironly{j}.png")
    axes[0, j].imshow(im.crop(jaw)); axes[0, j].set_title(lab, fontsize=12)
    axes[1, j].imshow(im); axes[1, j].set_title(lab, fontsize=12)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUT / "grid_fox_hair_only.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
src9 = storage.to_stored_size(Image.open(NORMAL / "normal_09_dark_skin.jpg").convert("RGB"))
full9, face9, hair9 = combo5_gpt.generate(src9, opts("남성", "20대", "강아지상").face.prompt)
(x1, y1, x2, y2), fw9 = combo5_gpt._face_box(src9)

VARS = [("현재", pipeline._postprocess_gpt(src9, full9, face9, hair9)),
        ("헤어만", hair_only(src9, full9, hair9)),
        ("헤어만 + 페더 5%", hair_only(src9, full9, hair9, feather=0.05)),
        ("헤어만 + 페더 10%", hair_only(src9, full9, hair9, feather=0.10))]
fig, axes = plt.subplots(1, len(VARS) + 1, figsize=(6 * (len(VARS) + 1), 8))
axes[0].imshow(src9); axes[0].set_title("src", fontsize=12)
for ax, (lab, im) in zip(axes[1:], VARS):
    im.save(OUT / f"normal_09_puppy_haironly_{lab}.png")
    ax.imshow(im); ax.set_title(lab, fontsize=12)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUT / "grid_09_hair_only.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
def skin_composite(src_, full_, hair_, dilate_px=0):
    cat = masks._category_mask(src_)
    skin = ((cat == 2) | (cat == 3)).astype(np.uint8) * 255
    skin = Image.fromarray(skin).resize(src_.size)
    skin = masks._morph(skin, cv2.MORPH_DILATE, dilate_px)
    scale = max(src_.size) / 1024
    sb = skin.filter(ImageFilter.GaussianBlur(settings.HAIR_BLUR * scale))
    comp = Image.composite(full_, src_, sb)  # 피부만 GPT
    hb = hair_.filter(ImageFilter.GaussianBlur(settings.HAIR_BLUR * scale))
    return Image.composite(src_, comp, hb)  # 헤어는 원본

CASES3 = [("salon_01 fox", src, full, hair, fw_face), ("09 puppy", src9, full9, hair9, fw9)]
DPX = [0.0, 0.02, 0.05]
fig, axes = plt.subplots(len(CASES3), len(DPX) + 1, figsize=(6 * (len(DPX) + 1), 8 * len(CASES3)))
for i, (lab, s_, f_, h_, fwv) in enumerate(CASES3):
    axes[i, 0].imshow(s_); axes[i, 0].set_title(f"{lab} src", fontsize=12)
    for ax, d in zip(axes[i, 1:], DPX):
        fin_ = skin_composite(s_, f_, h_, int(fwv * d))
        fin_.save(OUT / f"{lab.replace(' ', '_')}_skincomp{d}.png")
        ax.imshow(fin_); ax.set_title(f"skin dilate {d}", fontsize=12)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUT / "grid_skin_composite.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
for lab, s_, f_, h_, fwv in CASES3:
    fin_ = skin_composite(s_, f_, h_, int(fwv * 0.02))
    diff = np.abs(np.array(s_).astype(np.int16) - np.array(fin_).astype(np.int16)).max(axis=2) > 8
    hair_cls = masks._category_mask(s_) == 1
    hair_cls = np.array(Image.fromarray(hair_cls.astype(np.uint8) * 255).resize(s_.size)) > 127
    print(f"{lab}: 헤어 클래스 {hair_cls.sum()}px 중 바뀐 픽셀 {(diff & hair_cls).sum()}px ({100 * (diff & hair_cls).sum() / hair_cls.sum():.1f}%)")

In [ ]:
s_, f_, h_, fwv = src9, full9, hair9, fw9
fin9 = skin_composite(s_, f_, h_, int(fwv * 0.02))
(x1, y1, x2, y2), _ = combo5_gpt._face_box(s_)
fw = x2 - x1
bang = (x1 - fw // 6, max(0, y1 - fw // 6), x2 + fw // 6, y1 + fw // 2)
diff = np.abs(np.array(s_).astype(np.int16) - np.array(fin9).astype(np.int16)).max(axis=2) > 8
ov = np.array(s_).copy()
ov[diff] = (ov[diff] * 0.4 + np.array([255, 0, 0]) * 0.6).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(21, 8))
for ax, (lab, im) in zip(axes, [("src", s_), ("skin composite", fin9), ("changed=red", Image.fromarray(ov))]):
    ax.imshow(im.crop(bang)); ax.set_title(lab, fontsize=12); ax.axis("off")
plt.tight_layout()
plt.savefig(OUT / "grid_09_bang_diff.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
hair_raw = masks.build_hair_mask(s_, dilate=0)
hair_closed = masks._morph(hair_raw, cv2.MORPH_CLOSE, int(fwv * 0.123))
VARS = [("현재(눈썹 제외 있음)", h_), ("눈썹 제외 없음", hair_closed),
        ("눈썹 제외 없음 + 팽창 2%", masks._morph(hair_closed, cv2.MORPH_DILATE, int(fwv * 0.02)))]
brow = (x1 - fw // 8, y1 + fw // 6, x2 + fw // 8, y1 + fw // 2)
fig, axes = plt.subplots(2, len(VARS) + 1, figsize=(6 * (len(VARS) + 1), 10))
axes[0, 0].imshow(s_.crop(bang)); axes[0, 0].set_title("src 앞머리", fontsize=12)
axes[1, 0].imshow(s_.crop(brow)); axes[1, 0].set_title("src 눈썹", fontsize=12)
for j, (lab, hm_) in enumerate(VARS, start=1):
    r = skin_composite(s_, f_, hm_, int(fwv * 0.02))
    r.save(OUT / f"normal_09_puppy_browopt{j}.png")
    axes[0, j].imshow(r.crop(bang)); axes[0, j].set_title(lab, fontsize=12)
    axes[1, j].imshow(r.crop(brow)); axes[1, j].set_title(lab, fontsize=12)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUT / "grid_09_brow_option.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
from src.ai_engine.image_gen import compose

OUT5 = Path("/content/drive/MyDrive/saloncut_data/outputs/combo5_0825")
SALON_SET = ["salon_01_long_wave_brown", "salon_02_long_wave_dark", "salon_03_long_wave_ring",
             "salon_04_long_wave_black", "salon_05_short_bob_brown"]
STYLES = {"puppy": "강아지상", "cat": "고양이상", "fox": "여우상"}

for name in SALON_SET:
    s_ = storage.to_stored_size(Image.open(SALON / f"{name}.jpg").convert("RGB"))
    (x1, y1, x2, y2), fwv = combo5_gpt._face_box(s_)
    h_ = masks.build_hair_mask_gpt(s_, fwv)
    cat = masks._category_mask(s_)
    skin = Image.fromarray(((cat == 2) | (cat == 3)).astype(np.uint8) * 255).resize(s_.size)
    skin_d = masks._morph(skin, cv2.MORPH_DILATE, int(fwv * 0.02))
    ct_mask = masks.build_gen_mask(skin_d, h_)
    fig, axes = plt.subplots(1, len(STYLES) + 1, figsize=(6 * (len(STYLES) + 1), 7))
    axes[0].imshow(s_); axes[0].set_title(f"{name[:8]} src", fontsize=12)
    for ax, key in zip(axes[1:], STYLES):
        raw = Image.open(OUT5 / f"c5_{name}_{key}_gpt_v6_raw.png")
        f_ = s_.copy()
        f_.paste(raw.resize((x2 - x1, y2 - y1), Image.LANCZOS), (x1, y1))
        f_ = compose.color_transfer(f_, s_, ct_mask, alpha=settings.GPT_COLOR_ALPHA)
        fin_ = skin_composite(s_, f_, h_, int(fwv * 0.02))
        fin_.save(OUT / f"{name}_{key}_skincomp.png")
        ax.imshow(fin_); ax.set_title(key, fontsize=12)
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(OUT / f"grid_{name}_skincomp.png", dpi=100, bbox_inches="tight")
    plt.show()

In [ ]:
FOX_SET = ["salon_01_long_wave_brown", "salon_03_long_wave_ring"]
N = 3

for name in FOX_SET:
    s_ = storage.to_stored_size(Image.open(SALON / f"{name}.jpg").convert("RGB"))
    (x1, y1, x2, y2), fwv = combo5_gpt._face_box(s_)
    fw = x2 - x1
    jaw = (x1 - fw // 4, y1 + fw // 3, x2 + fw // 4, y2 + fw // 3)
    h_ = masks.build_hair_mask_gpt(s_, fwv)
    cat = masks._category_mask(s_)
    skin = Image.fromarray(((cat == 2) | (cat == 3)).astype(np.uint8) * 255).resize(s_.size)
    skin_d = masks._morph(skin, cv2.MORPH_DILATE, int(fwv * 0.02))
    ct_mask = masks.build_gen_mask(skin_d, h_)
    fig, axes = plt.subplots(2, N + 1, figsize=(6 * (N + 1), 13))
    axes[0, 0].imshow(s_); axes[0, 0].set_title(f"{name[:8]} src", fontsize=12)
    axes[1, 0].imshow(s_.crop(jaw)); axes[1, 0].set_title("src 턱", fontsize=12)
    for j in range(N):
        t = time.time()
        f_, _, _ = combo5_gpt.generate(s_, opts("여성", "20대", "여우상").face.prompt)
        f_ = compose.color_transfer(f_, s_, ct_mask, alpha=settings.GPT_COLOR_ALPHA)
        fin_ = skin_composite(s_, f_, h_, int(fwv * 0.02))
        fin_.save(OUT / f"{name}_fox_skin_r{j}.png")
        axes[0, j + 1].imshow(fin_); axes[0, j + 1].set_title(f"fox regen {j}  {time.time() - t:.0f}s", fontsize=12)
        axes[1, j + 1].imshow(fin_.crop(jaw)); axes[1, j + 1].set_title(f"regen {j} 턱", fontsize=12)
    for ax in axes.flat:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(OUT / f"grid_{name}_fox_skin_regen.png", dpi=100, bbox_inches="tight")
    plt.show()

In [ ]:
from src.ai_engine.image_gen import combo5_gpt, compose, masks

PAIRS = [("salon_01_long_wave_brown", SALON, "여성", "20대", "여우상"),
         ("normal_09_dark_skin", NORMAL, "남성", "20대", "강아지상")]
BROW = [0.0, 0.03, 0.048]
SKIN = [0.02, 0.03, 0.05]

store = {}
for name, folder, gender, age, style in PAIRS:
    s_ = storage.to_stored_size(Image.open(folder / f"{name}.jpg").convert("RGB"))
    f_, _, _ = combo5_gpt.generate(s_, opts(gender, age, style).face.prompt)
    _, fwv = combo5_gpt._face_box(s_)
    store[name] = (s_, f_, fwv)

def post(s_, f_, fwv, brow, skin_ratio):
    settings.BROW_EXCLUDE_RATIO = brow
    settings.GPT_SKIN_DILATE_RATIO = skin_ratio
    h_ = masks.build_hair_mask_gpt(s_, fwv)
    sk = masks.build_skin_mask(s_, fwv)
    return pipeline._postprocess_gpt(s_, f_, sk, h_)

for name, (s_, f_, fwv) in store.items():
    fig, axes = plt.subplots(len(BROW), len(SKIN) + 1, figsize=(6 * (len(SKIN) + 1), 7 * len(BROW)))
    for i, b in enumerate(BROW):
        axes[i, 0].imshow(s_); axes[i, 0].set_title(f"{name[:9]} src", fontsize=11)
        for ax, sr in zip(axes[i, 1:], SKIN):
            fin_ = post(s_, f_, fwv, b, sr)
            fin_.save(OUT / f"{name}_brow{b}_skin{sr}.png")
            ax.imshow(fin_); ax.set_title(f"brow {b} / skin {sr}", fontsize=11)
    for ax in axes.flat:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(OUT / f"grid_{name}_brow_skin.png", dpi=100, bbox_inches="tight")
    plt.show()
settings.BROW_EXCLUDE_RATIO, settings.GPT_SKIN_DILATE_RATIO = 0.048, 0.02

In [ ]:
%cd /content
import importlib
import os
import shutil
import sys
import time
from getpass import getpass
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
from google.colab import drive
from PIL import Image

drive.mount("/content/drive")

REPO = "/content/SalonCutAI"
shutil.rmtree(REPO, ignore_errors=True)
!git clone -q -b feat/combo5-gpt-v2 https://github.com/qja0707/SalonCutAI.git {REPO}
!pip install -q diffusers==0.39.0 transformers==5.14.1 peft==0.19.1 accelerate==1.14.0 \
    insightface onnxruntime mediapipe==1.0.0 opencv-contrib-python-headless \
    facexlib torchvision openai==2.53.0

os.environ["IMAGE_GEN_ENABLED"] = "1"
os.environ["SALON_STORAGE_DIR"] = "/content/storage"
os.environ["OPENAI_KEY"] = getpass("OpenAI API key: ")
sys.path.insert(0, f"{REPO}/backend")
importlib.invalidate_caches()

import mediapipe as mp
from src.ai_engine.image_gen import combo5_gpt, downloads, loader, pipeline, settings, storage
from src.schemas.face_swap import FacePromptOptions

downloads.ensure_models()
print("missing:", downloads.missing_files())
print("engine:", settings.PROMPT_MODE_ENGINE, "feather:", settings.GPT_PASTE_FEATHER_RATIO)

loader.get_face_app()
loader.get_landmarker()
loader.get_segmenter()
print("준비 완료")

In [ ]:
SALON = Path("/content/drive/MyDrive/saloncut_data/test_images/salon")
OUT = Path("/content/drive/MyDrive/saloncut_data/outputs/combo5_gpt_repo_0826")
OUT.mkdir(parents=True, exist_ok=True)


def opts(gender, age, style):
    prompt = FacePromptOptions(ethnicity="한국인", gender=gender, age=age, face_style=style, expression="무표정")
    return SimpleNamespace(face=SimpleNamespace(mode="prompt", prompt=prompt))


src_full = storage.to_stored_size(Image.open(SALON / "salon_01_long_wave_brown.jpg").convert("RGB"))
(bx1, by1, bx2, by2), fw = combo5_gpt._face_box(src_full)
cx = (bx1 + bx2) // 2
half_w = int(fw * 0.7)  # 폭을 얼굴 폭 1.4배로 좁혀 상자(1.6배)가 이미지보다 커지게
narrow = src_full.crop((cx - half_w, 0, cx + half_w, src_full.height))
(x1, y1, x2, y2), _ = combo5_gpt._face_box(narrow)
fw = int(fw)
print("narrow", narrow.size, "box", (x1, y1, x2, y2), "패딩:", x1 < 0 or x2 > narrow.width)

o = opts("여성", "20대", "고양이상")
res = {}
for f in [0.0, 0.05]:
    settings.GPT_PASTE_FEATHER_RATIO = f
    res[f] = pipeline._run_prompt_mode(narrow, o, seed=0)
    res[f].save(OUT / f"narrow_feather{f}.png")
settings.GPT_PASTE_FEATHER_RATIO = 0.05


def row_delta(img, y, band=6):
    a = np.array(img.convert("L")).astype(np.float32)
    return np.abs(a[y + band] - a[y - band]).mean()


band_box = (0, y2 - fw // 2, narrow.width, min(narrow.height, y2 + fw // 2))
fig, axes = plt.subplots(1, 3, figsize=(15, 7))
axes[0].imshow(narrow.crop(band_box)); axes[0].set_title("src (상자 아래 변 부근)", fontsize=11)
for ax, f in zip(axes[1:], [0.0, 0.05]):
    ax.imshow(res[f].crop(band_box))
    ax.set_title(f"feather {f}  경계 {row_delta(res[f], y2):.2f} / 주변 {row_delta(res[f], y2 + fw // 4):.2f}", fontsize=10)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUT / "grid_narrow_feather.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
def row_delta_center(img, y, band=6, frac=0.3):
    a = np.array(img.convert("L")).astype(np.float32)
    w = a.shape[1]
    c0, c1 = int(w * (0.5 - frac / 2)), int(w * (0.5 + frac / 2))
    return np.abs(a[y + band, c0:c1] - a[y - band, c0:c1]).mean()

for f in [0.0, 0.05]:
    print(f"feather {f}: 경계 {row_delta_center(res[f], y2):.2f} / 주변 {row_delta_center(res[f], y2 + fw // 4):.2f}")

In [ ]:
NORMAL = Path("/content/drive/MyDrive/saloncut_data/test_images/normal")
CASES = [("salon_03_long_wave_ring", SALON, "여성", "20대", "여우상"),
         ("salon_05_short_bob_brown", SALON, "여성", "20대", "강아지상"),
         ("normal_09_dark_skin", NORMAL, "남성", "20대", "고양이상")]

fig, axes = plt.subplots(len(CASES), 3, figsize=(15, 6 * len(CASES)))
for i, (name, folder, gender, age, style) in enumerate(CASES):
    full_img = storage.to_stored_size(Image.open(folder / f"{name}.jpg").convert("RGB"))
    (bx1, by1, bx2, by2), fw = combo5_gpt._face_box(full_img)
    cx = (bx1 + bx2) // 2
    half_w = int(fw * 0.7)
    nar = full_img.crop((max(0, cx - half_w), 0, min(full_img.width, cx + half_w), full_img.height))
    (x1, y1, x2, y2), _ = combo5_gpt._face_box(nar)
    fw = int(fw)
    print(f"{name[:9]} narrow {nar.size} box {(x1, y1, x2, y2)} 패딩 {x1 < 0 or x2 > nar.width}")
    band_box = (0, y2 - fw // 2, nar.width, min(nar.height, y2 + fw // 2))
    axes[i, 0].imshow(nar.crop(band_box)); axes[i, 0].set_title(f"{name[:9]} src", fontsize=10)
    for ax, f in zip(axes[i, 1:], [0.0, 0.05]):
        settings.GPT_PASTE_FEATHER_RATIO = f
        r = pipeline._run_prompt_mode(nar, opts(gender, age, style), seed=0)
        r.save(OUT / f"{name}_narrow_feather{f}.png")
        e, s = row_delta_center(r, y2), row_delta_center(r, y2 + fw // 4)
        ax.imshow(r.crop(band_box)); ax.set_title(f"feather {f}  edge {e:.2f} / around {s:.2f}", fontsize=10)
        print(f"   feather {f}: 경계 {e:.2f} / 주변 {s:.2f}")
settings.GPT_PASTE_FEATHER_RATIO = 0.05
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUT / "grid_narrow_feather_cases.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
_edit_orig = combo5_gpt._edit
_cache = {}

def _edit_cached(crop, rgba, prompt):
    if "raw" not in _cache:
        _cache["raw"] = _edit_orig(crop, rgba, prompt)
    return _cache["raw"]

combo5_gpt._edit = _edit_cached

FEATHERS = [0.0, 0.05, 0.10, 0.15]
CASES2 = [("normal_09_dark_skin", NORMAL, "남성", "20대", "고양이상"),
          ("salon_05_short_bob_brown", SALON, "여성", "20대", "강아지상")]

fig, axes = plt.subplots(len(CASES2), len(FEATHERS) + 1, figsize=(5 * (len(FEATHERS) + 1), 6 * len(CASES2)))
for i, (name, folder, gender, age, style) in enumerate(CASES2):
    _cache.clear()
    full_img = storage.to_stored_size(Image.open(folder / f"{name}.jpg").convert("RGB"))
    (bx1, by1, bx2, by2), fw = combo5_gpt._face_box(full_img)
    cx = (bx1 + bx2) // 2
    half_w = int(fw * 0.7)
    nar = full_img.crop((max(0, cx - half_w), 0, min(full_img.width, cx + half_w), full_img.height))
    (x1, y1, x2, y2), _ = combo5_gpt._face_box(nar)
    fw = int(fw)
    band_box = (0, y2 - fw // 2, nar.width, min(nar.height, y2 + fw // 2))
    axes[i, 0].imshow(nar.crop(band_box)); axes[i, 0].set_title(f"{name[:9]} src", fontsize=10)
    for ax, f in zip(axes[i, 1:], FEATHERS):
        settings.GPT_PASTE_FEATHER_RATIO = f
        r = pipeline._run_prompt_mode(nar, opts(gender, age, style), seed=0)
        r.save(OUT / f"{name}_narrow_sameraw_feather{f}.png")
        e, s = row_delta_center(r, y2), row_delta_center(r, y2 + fw // 4)
        ax.imshow(r.crop(band_box)); ax.set_title(f"feather {f}  edge {e:.2f} / around {s:.2f}", fontsize=10)
        print(f"{name[:9]} feather {f}: 경계 {e:.2f} / 주변 {s:.2f}")

combo5_gpt._edit = _edit_orig
settings.GPT_PASTE_FEATHER_RATIO = 0.05
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUT / "grid_narrow_feather_sweep.png", dpi=100, bbox_inches="tight")
plt.show()